# HumanEval OLMo 2 7B Instruct: Retokenization Length Scale

This notebook computes the retokenized length scale factor

$$\alpha_{\mu}(s) = \frac{L_{\mu}(s)}{L_0(s)} - 1$$

for HumanEval prompts under the saved nonzero retokenization runs for `allenai/OLMo-2-1124-7B-Instruct`.

Here `s` is only the prompt span that was retokenized. In the HumanEval retokenization experiment, raw `test_df.prompt` strings are passed to `batched_generate_legacy`, and `fiddle_tokens(..., no_prefix_suffix=True)` sets both prompt-wrapper lengths to zero. This excludes chat-template/prefix/suffix tokens from the length ratio.

The notebook prefers the saved `source_prompt_tokens` from the `_temp_from_retok` runs. If those are unavailable for a `p_retok`, it falls back to recovering the saved retokenized prompt prefix from the original run's `generation_tokens`, using the same decode-prefix method as `experiments/passat/retok/humaneval_temperature_from_retok.py`.

In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy import stats
from transformers import AutoTokenizer

REPO_ROOT = Path.cwd().resolve()
if REPO_ROOT.name == "figure_notebooks":
    REPO_ROOT = REPO_ROOT.parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

MODEL_NAME = "allenai/OLMo-2-1124-7B-Instruct"
MODEL_DIR_NAME = "allenai_OLMo-2-1124-7B-Instruct"
P_RETOK_VALUES = [0.2, 0.4, 0.6, 0.8, 1.0]
MAX_EXAMPLES = 164
UNBIASED_SIZE = 51

RESULT_DIR = (
    REPO_ROOT
    / "data"
    / "raw"
    / "Tokenizer_passK"
    / "results_passretok"
    / "humaneval"
    / MODEL_DIR_NAME
)
FIGURE_DIR = REPO_ROOT / "outputs" / "figures"
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

plt.rcParams.update(
    {
        "font.size": 11,
        "axes.spines.top": False,
        "axes.spines.right": False,
        "figure.dpi": 140,
        "savefig.bbox": "tight",
    }
)

RESULT_DIR

In [ ]:
def require_file(path: Path) -> Path:
    if not path.exists():
        raise FileNotFoundError(path)
    return path


def read_jsonl(path: Path) -> pd.DataFrame:
    return pd.read_json(require_file(path), lines=True)


def run_dir(p_retok: float, *, temp_from_retok: bool = False) -> Path:
    suffix = "_temp_from_retok" if temp_from_retok else ""
    return RESULT_DIR / f"retokp_{p_retok}_maxexamples_{MAX_EXAMPLES}_unbiasedsize_{UNBIASED_SIZE}{suffix}"


def prediction_path_for_p(p_retok: float) -> Path:
    temp_path = run_dir(p_retok, temp_from_retok=True) / "scored_predictions.jsonl"
    if temp_path.exists():
        return temp_path
    scored_path = run_dir(p_retok) / "scored_predictions.jsonl"
    if scored_path.exists():
        return scored_path
    return run_dir(p_retok) / "predictions.jsonl"


def strip_leading_special_tokens(token_ids: list[int], tokenizer) -> list[int]:
    pad_token_id = getattr(tokenizer, "pad_token_id", None)
    bos_token_id = getattr(tokenizer, "bos_token_id", None)
    special_prefix_ids = {token_id for token_id in (pad_token_id, bos_token_id) if token_id is not None}
    start = 0
    while start < len(token_ids) and int(token_ids[start]) in special_prefix_ids:
        start += 1
    return [int(token_id) for token_id in token_ids[start:]]


def recover_prompt_token_ids(row: pd.Series, tokenizer) -> list[int]:
    if "source_prompt_tokens" in row and isinstance(row["source_prompt_tokens"], list):
        return [int(token_id) for token_id in row["source_prompt_tokens"]]

    prompt = row["prompt"]
    token_ids = strip_leading_special_tokens(row["generation_tokens"], tokenizer)
    if not token_ids:
        raise ValueError(f"Empty generation_tokens for task_id={row['task_id']}.")

    for end in range(1, len(token_ids) + 1):
        decoded = tokenizer.decode(token_ids[:end], skip_special_tokens=True)
        if decoded == prompt:
            return token_ids[:end]

    preview = tokenizer.decode(token_ids[: min(len(token_ids), 128)], skip_special_tokens=True)
    raise ValueError(
        f"Could not recover retokenized prompt prefix for task_id={row['task_id']}.\n"
        f"Prompt starts with: {prompt[:200]!r}\n"
        f"Decoded token prefix starts with: {preview[:200]!r}"
    )


def canonical_prompt_length(prompt: str, tokenizer) -> int:
    token_ids = tokenizer.encode(prompt, add_special_tokens=True)
    return len(strip_leading_special_tokens(token_ids, tokenizer))


tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
    tokenizer.pad_token_id = tokenizer.eos_token_id

tokenizer

## Build the alpha table

`L_0(s)` is recomputed from the canonical prompt text with the same tokenizer setting used by generation (`add_special_tokens=True`, then leading pad/BOS stripped). `L_mu(s)` is the saved retokenized prompt-token sequence length.

In [ ]:
records = []

for p_retok in P_RETOK_VALUES:
    path = prediction_path_for_p(p_retok)
    df = read_jsonl(path)
    required = {"task_id", "prompt"}
    if "source_prompt_tokens" not in df.columns:
        required.add("generation_tokens")
    missing = required - set(df.columns)
    if missing:
        raise ValueError(f"{path} is missing required columns: {sorted(missing)}")

    for row in df.itertuples(index=False):
        row_s = pd.Series(row._asdict())
        source_prompt_tokens = recover_prompt_token_ids(row_s, tokenizer)
        canonical_length = canonical_prompt_length(row_s["prompt"], tokenizer)
        retok_length = len(source_prompt_tokens)
        records.append(
            {
                "task_id": row_s["task_id"],
                "p_retok": p_retok,
                "canonical_length": canonical_length,
                "retok_length": retok_length,
                "alpha": retok_length / canonical_length - 1.0,
                "source_path": str(path.relative_to(REPO_ROOT)),
            }
        )

alpha_df = pd.DataFrame.from_records(records)
alpha_df["sample_index"] = alpha_df.groupby(["p_retok", "task_id"]).cumcount()
alpha_df = alpha_df[
    ["task_id", "p_retok", "sample_index", "canonical_length", "retok_length", "alpha", "source_path"]
]

alpha_df.head()

In [ ]:
alpha_df

In [ ]:
task_summary

In [ ]:
summary = (
    alpha_df.groupby("p_retok", sort=True)
    .agg(
        num_samples=("alpha", "size"),
        num_tasks=("task_id", "nunique"),
        mean=("alpha", "mean"),
        std=("alpha", "std"),
        median=("alpha", "median"),
    )
    .reset_index()
)
summary["sem"] = summary["std"] / np.sqrt(summary["num_samples"])
summary["ci95"] = 1.96 * summary["sem"]

task_summary = (
    alpha_df.groupby(["task_id", "p_retok"], sort=True)["alpha"]
    .mean()
    .rename("task_mean_alpha")
    .reset_index()
)

def stat_result_value(result, field: str, index: int) -> float:
    if hasattr(result, field):
        return float(getattr(result, field))
    return float(result[index])


pooled_pearson = stats.pearsonr(alpha_df["p_retok"], alpha_df["alpha"])
pooled_spearman = stats.spearmanr(alpha_df["p_retok"], alpha_df["alpha"])
task_mean_pearson = stats.pearsonr(task_summary["p_retok"], task_summary["task_mean_alpha"])
task_mean_spearman = stats.spearmanr(task_summary["p_retok"], task_summary["task_mean_alpha"])
pooled_corr = stat_result_value(pooled_pearson, "statistic", 0)
task_mean_corr = stat_result_value(task_mean_pearson, "statistic", 0)

correlation_table = pd.DataFrame(
    [
        {
            "analysis": "pooled samples",
            "method": "Pearson r",
            "statistic": stat_result_value(pooled_pearson, "statistic", 0),
            "p_value": stat_result_value(pooled_pearson, "pvalue", 1),
            "note": "descriptive; samples share task prompts",
        },
        {
            "analysis": "pooled samples",
            "method": "Spearman rho",
            "statistic": stat_result_value(pooled_spearman, "statistic", 0),
            "p_value": stat_result_value(pooled_spearman, "pvalue", 1),
            "note": "descriptive monotonic association",
        },
        {
            "analysis": "task means",
            "method": "Pearson r",
            "statistic": stat_result_value(task_mean_pearson, "statistic", 0),
            "p_value": stat_result_value(task_mean_pearson, "pvalue", 1),
            "note": "one mean alpha per task and p_retok",
        },
        {
            "analysis": "task means",
            "method": "Spearman rho",
            "statistic": stat_result_value(task_mean_spearman, "statistic", 0),
            "p_value": stat_result_value(task_mean_spearman, "pvalue", 1),
            "note": "one mean alpha per task and p_retok",
        },
    ]
)

display(summary)
display(correlation_table)

## Statistical tests

The primary inferential statistic is a within-task linear trend: regress `alpha` on `p_retok` after demeaning both variables within each HumanEval task. This controls for task-level prompt length and tokenization differences. The table reports a cluster-robust p-value by task and a permutation p-value from shuffling `p_retok` labels within each task.

The chi-square tests are secondary diagnostics because they bin the continuous `alpha` values.

In [ ]:
def within_task_slope(df: pd.DataFrame, *, y_col: str = "alpha", x_col: str = "p_retok") -> dict:
    work = df[["task_id", x_col, y_col]].copy()
    x = work[x_col].to_numpy(dtype=float)
    y = work[y_col].to_numpy(dtype=float)
    x_bar = work.groupby("task_id")[x_col].transform("mean").to_numpy(dtype=float)
    y_bar = work.groupby("task_id")[y_col].transform("mean").to_numpy(dtype=float)
    x_within = x - x_bar
    y_within = y - y_bar
    sxx = float(np.dot(x_within, x_within))
    slope = float(np.dot(x_within, y_within) / sxx)
    residual = y_within - slope * x_within

    num_obs = len(work)
    num_tasks = work["task_id"].nunique()
    num_parameters = num_tasks + 1
    df_resid = num_obs - num_parameters

    cluster_scores = pd.DataFrame(
        {"task_id": work["task_id"], "score": x_within * residual}
    ).groupby("task_id")["score"].sum()
    meat = float(np.dot(cluster_scores, cluster_scores))
    small_sample = (num_tasks / (num_tasks - 1)) * ((num_obs - 1) / df_resid)
    cluster_var = small_sample * meat / (sxx ** 2)
    cluster_se = float(np.sqrt(cluster_var))
    t_stat = slope / cluster_se
    p_value = float(2 * stats.t.sf(abs(t_stat), df=num_tasks - 1))
    ci_mult = stats.t.ppf(0.975, df=num_tasks - 1)

    return {
        "slope": slope,
        "cluster_se": cluster_se,
        "t_stat": float(t_stat),
        "p_value": p_value,
        "ci95_low": float(slope - ci_mult * cluster_se),
        "ci95_high": float(slope + ci_mult * cluster_se),
        "num_obs": num_obs,
        "num_tasks": num_tasks,
    }


def permutation_p_value_for_within_slope(
    df: pd.DataFrame,
    *,
    observed_slope: float,
    n_permutations: int = 2_000,
    random_seed: int = 0,
) -> float:
    rng = np.random.default_rng(random_seed)
    permuted = df[["task_id", "p_retok", "alpha"]].copy()
    original_p_values = permuted["p_retok"].to_numpy(copy=True)
    task_indices = [group.index.to_numpy() for _, group in permuted.groupby("task_id", sort=False)]
    extreme = 0

    for _ in range(n_permutations):
        p_values = original_p_values.copy()
        for idx in task_indices:
            p_values[idx] = rng.permutation(p_values[idx])
        permuted["p_retok"] = p_values
        permuted_slope = within_task_slope(permuted)["slope"]
        if abs(permuted_slope) >= abs(observed_slope):
            extreme += 1

    return (extreme + 1) / (n_permutations + 1)


trend = within_task_slope(alpha_df)
permutation_p = permutation_p_value_for_within_slope(alpha_df, observed_slope=trend["slope"])

trend_table = pd.DataFrame(
    [
        {
            "analysis": "task fixed-effect trend",
            "statistic": "slope of alpha on p_retok",
            "estimate": trend["slope"],
            "std_error": trend["cluster_se"],
            "ci95_low": trend["ci95_low"],
            "ci95_high": trend["ci95_high"],
            "p_value": trend["p_value"],
            "note": "cluster-robust by task",
        },
        {
            "analysis": "task fixed-effect trend",
            "statistic": "permutation p-value",
            "estimate": trend["slope"],
            "std_error": np.nan,
            "ci95_low": np.nan,
            "ci95_high": np.nan,
            "p_value": permutation_p,
            "note": "p_retok shuffled within task",
        },
    ]
)

display(trend_table)

In [ ]:
expanded_table = pd.crosstab(alpha_df["p_retok"], alpha_df["alpha"] > 0)
expanded_chi2, expanded_p, expanded_dof, _ = stats.chi2_contingency(expanded_table)

alpha_df = alpha_df.copy()
alpha_df["alpha_quartile"] = pd.qcut(alpha_df["alpha"], q=4, labels=False, duplicates="drop")
quartile_table = pd.crosstab(alpha_df["p_retok"], alpha_df["alpha_quartile"])
quartile_chi2, quartile_p, quartile_dof, _ = stats.chi2_contingency(quartile_table)

chi_square_table = pd.DataFrame(
    [
        {
            "analysis": "p_retok x expanded",
            "definition": "expanded = alpha > 0",
            "chi2": expanded_chi2,
            "dof": expanded_dof,
            "p_value": expanded_p,
        },
        {
            "analysis": "p_retok x alpha quartile",
            "definition": "alpha split into empirical quartiles",
            "chi2": quartile_chi2,
            "dof": quartile_dof,
            "p_value": quartile_p,
        },
    ]
)

display(chi_square_table)
display(expanded_table)
display(quartile_table)

## Mean alpha versus retokenization probability

In [ ]:
fig, ax = plt.subplots(figsize=(5.4, 3.6))
ax.errorbar(
    summary["p_retok"],
    summary["mean"],
    yerr=summary["ci95"],
    marker="o",
    linewidth=1.8,
    capsize=3,
    color="#1f77b4",
)
ax.axhline(0.0, color="0.35", linewidth=0.8, linestyle="--")
ax.set_xlabel("retokenization probability $p_{retok}$")
ax.set_ylabel(r"mean length scale $\alpha_{\mu}(s)$")
ax.set_title("HumanEval OLMo 2 7B Instruct")
ax.text(
    0.03,
    0.97,
    f"pooled r = {pooled_corr:.3f}",
    transform=ax.transAxes,
    ha="left",
    va="top",
    fontsize=10,
)
fig.savefig(FIGURE_DIR / "humaneval_olmo2_alpha_retok_mean.svg")
fig.savefig(FIGURE_DIR / "humaneval_olmo2_alpha_retok_mean.png", dpi=300)
plt.show()

## Distribution of alpha by p

In [ ]:
fig, ax = plt.subplots(figsize=(6.2, 3.6))
data_by_p = [alpha_df.loc[alpha_df["p_retok"] == p, "alpha"].to_numpy() for p in P_RETOK_VALUES]
parts = ax.violinplot(data_by_p, positions=P_RETOK_VALUES, widths=0.09, showextrema=False)
for body in parts["bodies"]:
    body.set_facecolor("#9ecae1")
    body.set_edgecolor("#1f77b4")
    body.set_alpha(0.7)

ax.boxplot(
    data_by_p,
    positions=P_RETOK_VALUES,
    widths=0.045,
    patch_artist=True,
    showfliers=False,
    boxprops={"facecolor": "white", "edgecolor": "0.2", "linewidth": 0.8},
    medianprops={"color": "0.1", "linewidth": 1.2},
    whiskerprops={"color": "0.25", "linewidth": 0.8},
    capprops={"color": "0.25", "linewidth": 0.8},
)
ax.axhline(0.0, color="0.35", linewidth=0.8, linestyle="--")
ax.set_xticks(P_RETOK_VALUES)
ax.set_xlabel("retokenization probability $p_{retok}$")
ax.set_ylabel(r"$\alpha_{\mu}(s)$")
ax.set_title("Distribution over saved retokenized HumanEval prompts")
fig.savefig(FIGURE_DIR / "humaneval_olmo2_alpha_retok_distribution.svg")
fig.savefig(FIGURE_DIR / "humaneval_olmo2_alpha_retok_distribution.png", dpi=300)
plt.show()

## Per-task mean alpha trends

In [ ]:
fig, ax = plt.subplots(figsize=(5.8, 3.8))
for task_id, group in task_summary.groupby("task_id"):
    ax.plot(group["p_retok"], group["task_mean_alpha"], color="0.65", linewidth=0.5, alpha=0.28)

task_mean_by_p = (
    task_summary.groupby("p_retok", sort=True)["task_mean_alpha"]
    .agg(mean="mean", std="std", n="size")
    .reset_index()
)
task_mean_by_p["ci95"] = 1.96 * task_mean_by_p["std"] / np.sqrt(task_mean_by_p["n"])
ax.errorbar(
    task_mean_by_p["p_retok"],
    task_mean_by_p["mean"],
    yerr=task_mean_by_p["ci95"],
    marker="o",
    linewidth=2.0,
    capsize=3,
    color="#d62728",
    label="mean over tasks",
)
ax.axhline(0.0, color="0.35", linewidth=0.8, linestyle="--")
ax.set_xlabel("retokenization probability $p_{retok}$")
ax.set_ylabel(r"task mean $\alpha_{\mu}(s)$")
ax.set_title("Task-level alpha trends")
ax.legend(frameon=False)
fig.savefig(FIGURE_DIR / "humaneval_olmo2_alpha_retok_task_trends.svg")
fig.savefig(FIGURE_DIR / "humaneval_olmo2_alpha_retok_task_trends.png", dpi=300)
plt.show()